In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from mlforecast import MLForecast
from tinyconformal.series import ConformalQuantileTimeSeriesRegressor
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression


In [2]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

# LigthGBM

In [3]:

#def create_mlforecast_multiquantile():
#        import lightgbm as lgb
#
#     # Define 90% and 50% prediction interval quantile LightGBM models
#     models = {
#         "LGBM-lo-90": lgb.LGBMRegressor(objective="quantile", alpha=0.05, random_state=42),
#         "LGBM-hi-90": lgb.LGBMRegressor(objective="quantile", alpha=0.95, random_state=42),
#         "LGBM-lo-50": lgb.LGBMRegressor(objective="quantile", alpha=0.25, random_state=42),
#         "LGBM-hi-50": lgb.LGBMRegressor(objective="quantile", alpha=0.75, random_state=42),
#     }
#     return MLForecast(
#         models=models,
#         freq="MS",
#         lags=[1, 7],
#     )
#models = create_mlforecast_multiquantile()

# RandomForestQuantileRegressor

In [4]:

class QuantileRF(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        quantile=0.5,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42,
        **kwargs,
    ):
        self.quantile = quantile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.kwargs = kwargs

    def fit(self, X, y):
        self.model_ = RandomForestQuantileRegressor(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            **self.kwargs,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X, quantiles=self.quantile)

In [5]:
def model_callable():
    models = {
        "RF-lo-90": QuantileRF(
            quantile=0.05, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-hi-90": QuantileRF(
            quantile=0.95, n_estimators=200, max_depth=10, random_state=42
        ),
        "RF-lo-50": QuantileRF(
            quantile=0.25, n_estimators=100, max_depth=8, random_state=42
        ),
        "RF-hi-50": QuantileRF(
            quantile=0.75, n_estimators=100, max_depth=8, random_state=42
        ),
        "RF-50": QuantileRF(
            quantile=0.50, n_estimators=100, max_depth=8, random_state=42
        ),
        #"LinearRegression": LinearRegression()
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )


models = model_callable()

In [6]:
cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     intervals={("RF-lo-90", "RF-hi-90"): "RF-50"},
     n_windows=7,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,intervals,"{('RF-lo-90', ...): 'RF-50'}"
,n_windows,7
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [7]:
cqr.predict_interval(h=7, X_df=test)

/home/heylucasleao/tinyconformal/tinyconformal/series/tscqr.py:475: RuntimeWarning: The requested coverage is not attainable with the available calibration sample (n=7); the empirical quantile level will be clipped to the largest observed score.
  q_level = self._sample_correction(self._validate_alpha(alpha))


,unique_id,ds,RF-lo-90,RF-hi-90,RF-lo-50,RF-hi-50,RF-50,RF-lo-90-cqr,RF-hi-90-cqr,RF-50-cqr
0,1,1960-01-01,337.00,436.5,363.0,406.00,396.0,331.30,442.20,397.059799
1,1,1960-02-01,305.00,548.0,353.0,406.00,405.0,303.25,549.75,404.690329
2,1,1960-03-01,301.00,559.0,360.0,406.00,396.0,259.00,601.00,384.930233
3,1,1960-04-01,301.00,559.0,342.0,405.00,405.0,260.00,600.00,397.054264
4,1,1960-05-01,300.15,559.0,337.0,405.00,379.5,265.15,594.00,365.958374
5,1,1960-06-01,277.00,559.0,336.0,405.00,361.0,250.85,585.15,350.428723
6,1,1960-07-01,276.10,559.0,318.0,398.25,363.0,205.10,630.00,335.618947


In [8]:
cqr.evaluate(test, h=12)

/home/heylucasleao/tinyconformal/tinyconformal/series/tscqr.py:475: RuntimeWarning: The requested coverage is not attainable with the available calibration sample (n=7); the empirical quantile level will be clipped to the largest observed score.
  q_level = self._sample_correction(self._validate_alpha(alpha))


,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,RF,90%,0.1,0.833,282.354,465.688
1,RF-cqr,90%,0.1,1.000,317.288,317.288


# GradientBoostingRegressor & HistGradientBoostingRegressor

In [9]:
def model_callable():
    models = {
        "GBR-lo-90": GradientBoostingRegressor(loss="quantile", alpha=0.05, n_estimators=100, random_state=42),
        "GBR-hi-90": GradientBoostingRegressor(loss="quantile", alpha=0.95, n_estimators=100, random_state=42),
        "HGB-lo-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.05, max_iter=100, random_state=42),
        "HGB-hi-90": HistGradientBoostingRegressor(loss="quantile", quantile=0.95, max_iter=100, random_state=42),
    }
    return MLForecast(
        models=models,
        lags=[1, 2, 7],
        freq="MS",
    )

models = model_callable()

In [15]:
cqr = ConformalQuantileTimeSeriesRegressor(
     learner=models,
     horizon=12,
     intervals=[
         ("GBR-lo-90", "GBR-hi-90"),
         ("HGB-lo-90", "HGB-hi-90"),
     ],
     n_windows=10,
 )
cqr.fit(train, static_features=[])

,learner,MLForecast(mo...num_threads=1)
,horizon,12
,intervals,"[('GBR-lo-90', ...), ('HGB-lo-90', ...)]"
,n_windows,10
,id_col,'unique_id'
,time_col,'ds'
,target_col,'y'


In [22]:
cqr.predict_interval(h=12)

,unique_id,ds,GBR-lo-90,GBR-hi-90,HGB-lo-90,HGB-hi-90,GBR-lo-90-cqr,GBR-hi-90-cqr,HGB-lo-90-cqr,HGB-hi-90-cqr
0,1,1960-01-01,354.096291,434.369949,351.696226,548.200517,293.040622,495.425617,359.896226,540.000517
1,1,1960-02-01,301.680667,494.753113,280.350243,548.200517,302.759410,493.674371,293.350243,535.200517
2,1,1960-03-01,286.781772,527.501404,228.506346,548.200517,247.860515,566.422662,220.506346,556.200517
3,1,1960-04-01,279.363798,463.989313,217.483443,548.200517,228.894864,514.458248,214.345061,551.338899
4,1,1960-05-01,279.363798,452.503306,217.676035,548.200517,238.246763,493.620341,215.676035,550.200517
5,1,1960-06-01,277.909028,493.730766,217.676035,548.200517,201.409273,570.230520,191.070682,574.805870
6,1,1960-07-01,279.363798,549.529707,217.676035,548.200517,153.243805,675.649700,146.690066,619.186486
7,1,1960-08-01,277.909028,463.989313,218.292664,548.200517,160.532320,581.366021,150.488007,616.005174
8,1,1960-09-01,255.048351,452.503306,215.856502,548.200517,178.448348,529.103308,196.870533,567.186486
9,1,1960-10-01,229.407800,493.453113,207.942414,548.200517,162.289550,560.571364,209.942414,546.200517


In [12]:
cqr.evaluate(test, h=12)

,model,level,alpha,coverage_rate,interval_width_mean,mwis
0,GBR,90%,0.1,0.583,219.015,770.254
1,GBR-cqr,90%,0.1,0.917,336.957,378.013
2,HGB,90%,0.1,0.833,321.509,540.841
3,HGB-cqr,90%,0.1,0.917,334.220,338.909
